# Nutrisense-AI — Disease Risk Models (XGBoost)
## Capstone Project | African Leadership University

**Author:** Jean Jabo  
**Data:** Rwanda DHS 2019-20 (real biomarker outcomes + calibrated dietary features)  
**Models:** XGBoost binary classifiers — Anemia · Diabetes · Overweight

**Before running:**
1. Right panel → Add Data → `nutrisense-dhs-rwanda` (your private dataset)
2. Add-ons → Secrets → add `HF_TOKEN` with your HuggingFace write token
3. Internet → **On**

In [ ]:
!pip install shap xgboost huggingface_hub -q 2>/dev/null

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import shap, joblib
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

SAVE_DIR = '/kaggle/working'
print('Setup complete.')

## Part 1 — Data Loading

In [ ]:
FEATURES = [
    'age', 'sex',
    'energy_kcal', 'protein_g', 'fat_g', 'carb_g',
    'iron_mg', 'vitC_mg', 'vitA_mcg',
    'fiber_g', 'sugar_g', 'calcium_mg', 'zinc_mg', 'sodium_mg',
]
TARGETS = ['anemia', 'diabetes', 'overweight']

DHS_PATH = '/kaggle/input/nutrisense-dhs-rwanda/nutrisense_dhs.csv'

if os.path.exists(DHS_PATH):
    df = pd.read_csv(DHS_PATH)
    DATA_SOURCE = 'dhs'
    print(f'✅ Loaded Rwanda DHS 2019-20: {df.shape}')
else:
    raise FileNotFoundError(
        'DHS dataset not found. Add nutrisense-dhs-rwanda via right panel → Add Data.'
    )

df = df.dropna(subset=FEATURES + TARGETS)
print(f'Rows   : {len(df):,}')
print(f'Anemia : {df.anemia.mean()*100:.1f}%  '
      f'Diabetes : {df.diabetes.mean()*100:.1f}%  '
      f'Overweight : {df.overweight.mean()*100:.1f}%')

## Part 2 — Exploratory Data Analysis

In [ ]:
# Feature distributions
key_feats = ['age', 'energy_kcal', 'iron_mg', 'protein_g', 'sugar_g', 'carb_g']
palette   = sns.color_palette('viridis', len(key_feats))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat, color in zip(axes.flatten(), key_feats, palette):
    ax.hist(df[feat], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[feat].mean(),   color='red',    linestyle='--', label=f'Mean {df[feat].mean():.1f}')
    ax.axvline(df[feat].median(), color='orange', linestyle=':',  label=f'Median {df[feat].median():.1f}')
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions — Rwanda DHS 2019-20', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/10_feature_distributions.png', dpi=120)
plt.show()
print('Saved → 10_feature_distributions.png')

In [ ]:
# Correlation heatmap
corr = df[FEATURES].corr()
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, square=True,
            xticklabels=[f.replace('_','\n') for f in FEATURES],
            yticklabels=[f.replace('_','\n') for f in FEATURES])
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/11_correlation_heatmap.png', dpi=120)
plt.show()
print('Saved → 11_correlation_heatmap.png')

In [ ]:
# Disease prevalence
COLORS = {'anemia': '#4C72B0', 'diabetes': '#C44E52', 'overweight': '#55A868'}
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, tgt in zip(axes, TARGETS):
    counts = df[tgt].value_counts().sort_index()
    bars   = ax.bar(['Negative','Positive'], counts.values,
                    color=[COLORS[tgt], COLORS[tgt]], edgecolor='white', linewidth=1.5)
    bars[0].set_alpha(0.35)
    bars[1].set_alpha(0.90)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=10)
    ax.set_title(tgt.upper(), fontweight='bold', fontsize=13, color=COLORS[tgt])
    ax.set_ylabel('Count')

plt.suptitle('Disease Label Distribution — Rwanda DHS 2019-20', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/12_class_imbalance.png', dpi=120)
plt.show()
print('Saved → 12_class_imbalance.png')

## Part 3 — XGBoost Training (5-fold CV)

In [ ]:
X = df[FEATURES].astype(float).values
Y = df[TARGETS].values
X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=42)

XGB_PARAMS = dict(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='auc', random_state=42, n_jobs=-1,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models_xgb = {}; explainers = {}; cv_results = {}

for i, tgt in enumerate(TARGETS):
    y_tr, y_te = Y_tr[:, i], Y_te[:, i]
    sw         = compute_sample_weight('balanced', y_tr)
    print(f'\n{"="*52}\n{tgt.upper()}  (train pos rate {y_tr.mean()*100:.1f}%)')

    make_pipe = lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    XGBClassifier(**XGB_PARAMS))
    ])

    aucs, aps = [], []
    for fold, (tr, va) in enumerate(cv.split(X_tr, y_tr)):
        p = make_pipe()
        p.fit(X_tr[tr], y_tr[tr], clf__sample_weight=sw[tr])
        prob = p.predict_proba(X_tr[va])[:, 1]
        aucs.append(roc_auc_score(y_tr[va], prob))
        aps.append(average_precision_score(y_tr[va], prob))
        print(f'  Fold {fold+1}: AUC {aucs[-1]:.3f}  AP {aps[-1]:.3f}')

    print(f'  CV  AUC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}  |  AP {np.mean(aps):.3f}')

    pipe = make_pipe()
    pipe.fit(X_tr, y_tr, clf__sample_weight=sw)
    models_xgb[tgt] = pipe
    explainers[tgt] = shap.TreeExplainer(pipe.named_steps['clf'])
    cv_results[tgt] = {'roc_auc': float(np.mean(aucs)), 'avg_prec': float(np.mean(aps)),
                        'std': float(np.std(aucs))}

print('\n✅ All 3 models trained.')

## Part 4 — Evaluation

In [ ]:
# ROC + Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for tgt in TARGETS:
    y_te   = Y_te[:, TARGETS.index(tgt)]
    prob   = models_xgb[tgt].predict_proba(X_te)[:, 1]
    color  = COLORS[tgt]
    auc    = roc_auc_score(y_te, prob)
    ap     = average_precision_score(y_te, prob)
    fpr, tpr, _ = roc_curve(y_te, prob)
    prec, rec, _ = precision_recall_curve(y_te, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5, label=f'{tgt.title()} (AUC={auc:.3f})')
    axes[1].plot(rec, prec, color=color, linewidth=2.5, label=f'{tgt.title()} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set(title='ROC Curves', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Precision-Recall Curves', xlabel='Recall', ylabel='Precision')
axes[1].legend(); axes[1].grid(alpha=0.3)
for ax in axes: ax.set_xlim(0,1); ax.set_ylim(0,1.02)

plt.suptitle('XGBoost Risk Models — Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/13_roc_pr_curves.png', dpi=120)
plt.show()
print('Saved → 13_roc_pr_curves.png')

In [ ]:
# Confusion matrices
THRESHOLDS = {'anemia': 0.40, 'diabetes': 0.35, 'overweight': 0.50}
fig, axes  = plt.subplots(1, 3, figsize=(16, 5))
for ax, tgt in zip(axes, TARGETS):
    y_te  = Y_te[:, TARGETS.index(tgt)]
    prob  = models_xgb[tgt].predict_proba(X_te)[:, 1]
    preds = (prob >= THRESHOLDS[tgt]).astype(int)
    cm    = confusion_matrix(y_te, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'],
                annot_kws={'size': 14})
    ax.set_title(tgt.upper(), fontweight='bold', fontsize=12, color=COLORS[tgt])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/14_confusion_matrices.png', dpi=120)
plt.show()
print('Saved → 14_confusion_matrices.png')

In [ ]:
# SHAP feature importance
scaler_fit = StandardScaler().fit(X_tr)
X_sample   = scaler_fit.transform(X_te[:300])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
shap_top  = {}
for ax, tgt in zip(axes, TARGETS):
    sv        = explainers[tgt].shap_values(X_sample)
    mean_shap = np.abs(sv).mean(0)
    order     = mean_shap.argsort()[::-1][:10]
    feat_lbls = [FEATURES[j].replace('_',' ') for j in order[::-1]]
    ax.barh(feat_lbls, mean_shap[order[::-1]], color=COLORS[tgt], alpha=0.85)
    ax.set_title(f'SHAP — {tgt.upper()}', fontweight='bold', color=COLORS[tgt])
    ax.set_xlabel('Mean |SHAP value|')
    shap_top[tgt] = [{'f': FEATURES[j], 'v': round(float(np.mean(sv[:,j])),3)} for j in order[:5]]

plt.suptitle('Feature Importance via SHAP (top 10)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/15_shap_importance.png', dpi=120)
plt.show()
print('Saved → 15_shap_importance.png')

In [ ]:
# CV results summary table
fig, ax = plt.subplots(figsize=(9, 3))
ax.axis('off')
tbl_data = [[t.upper(),
             f"{cv_results[t]['roc_auc']:.3f}",
             f"± {cv_results[t]['std']:.3f}",
             f"{cv_results[t]['avg_prec']:.3f}"]
            for t in TARGETS]
tbl = ax.table(cellText=tbl_data,
               colLabels=['Disease', 'ROC-AUC', 'Std', 'Avg Precision'],
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(13); tbl.scale(1.5, 2.2)
plt.title('5-Fold Cross-Validation Results — Rwanda DHS 2019-20', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/17_cv_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → 17_cv_results.png')

## Part 5 — Save & Deploy to HuggingFace

In [ ]:
# Save XGBoost bundle
BUNDLE_PATH = f'{SAVE_DIR}/nutrisense_model.joblib'
bundle = {
    'features'   : FEATURES,
    'models'     : models_xgb,
    'explainers' : explainers,
    'shap_top'   : shap_top,
    'thresholds' : THRESHOLDS,
    'data_source': DATA_SOURCE,
    'label_definitions': {
        'anemia'    : 'Hb < 12 g/dL (women) / < 13 g/dL (men) [WHO 2011]',
        'overweight': 'BMI >= 25 [WHO]',
        'diabetes'  : 'HbA1c >= 6.5% [ADA 2023]',
    },
    'cv_scores': cv_results,
}
joblib.dump(bundle, BUNDLE_PATH)
print(f'Bundle saved → {BUNDLE_PATH}  ({os.path.getsize(BUNDLE_PATH)/1e6:.1f} MB)')

In [ ]:
# Push to HuggingFace Space
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
SPACE_ID = 'JeanJabo/nutrisense-api'

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()

print(f'Pushing nutrisense_model.joblib to {SPACE_ID} ...')
api.upload_file(
    path_or_fileobj=BUNDLE_PATH,
    path_in_repo='nutrisense_model.joblib',
    repo_id=SPACE_ID,
    repo_type='space',
    token=HF_TOKEN,
)
print(f'✅ Done → https://huggingface.co/spaces/{SPACE_ID}')

In [ ]:
# Final verification
OUTPUT_FILES = {
    '10_feature_distributions.png' : 'Feature distributions',
    '11_correlation_heatmap.png'   : 'Correlation heatmap',
    '12_class_imbalance.png'       : 'Disease prevalence',
    '13_roc_pr_curves.png'         : 'ROC and PR curves',
    '14_confusion_matrices.png'    : 'Confusion matrices',
    '15_shap_importance.png'       : 'SHAP feature importance',
    '17_cv_results.png'            : 'CV results table',
    'nutrisense_model.joblib'      : 'XGBoost model bundle',
}

print('=' * 55)
print('OUTPUT VERIFICATION')
print('=' * 55)
for fname, desc in OUTPUT_FILES.items():
    path   = f'{SAVE_DIR}/{fname}'
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else ''
    print(f'  {"✅" if exists else "❌"}  {desc:35s}  {size}')

print()
print(f'Data source : {DATA_SOURCE.upper()} (Rwanda DHS 2019-20)')
print(f'XGBoost AUCs: { {k: f"{v["roc_auc"]:.3f}" for k, v in cv_results.items()} }')
print('\n✅ ALL DONE')